In [1]:
import sys, torch
print(sys.executable)
print(torch.__version__, torch.cuda.is_available())

c:\Users\aeasa\anaconda3\envs\circmtl\python.exe
2.14.1+cu126 True


In [17]:
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd().parent.parent                                        # notebook runs from notebooks/
with open(PROJECT_ROOT / "configs" / "paths.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
PROCESSED_DIR = PROJECT_ROOT / cfg["outputs"]["processed_dir"]

In [18]:
# 06_embeddings - Cell 0: setup, model, tokenizer checks, embedding functions, timing
import json, time
import numpy as np
import pandas as pd
import torch
import fm

D = PROCESSED_DIR / "dataset_v1" 
OUT = D / "RNA-FM_embedding_output"; OUT.mkdir(exist_ok=True)

W, S, LAYER, BATCH = 1022, 511, 12, 4            # window, stride, last layer, windows per forward pass
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

seq = pd.read_csv(D / "sequences.csv", keep_default_na=False)
circ = pd.read_csv(D / "circrnas.csv", keep_default_na=False)
assert seq["circbase_id"].tolist() == circ["circbase_id"].tolist(), "row order differs from circrnas.csv"
assert (seq["sequence"].str.len() == seq["seq_len"]).all(), "sequences are not full length"

FM_WEIGHTS = PROJECT_ROOT / cfg["models"]["rnafm_weights"]                 # local weights (no download)
assert FM_WEIGHTS.exists(), f"RNA-FM weights not found: {FM_WEIGHTS}"

import argparse
torch.serialization.add_safe_globals([argparse.Namespace])

model, alphabet = fm.pretrained.rna_fm_t12(model_location=str(FM_WEIGHTS))
model = model.to(device).eval()
conv = alphabet.get_batch_converter()

# Tokenizer checks: U must be known; one start and one end token around the sequence
for t in ["ACGU", "ACGT"]:
    _, _, tok = conv([("t", t)])
    print(f"{t} -> {tok.tolist()} | unknown tokens: {int((tok == alphabet.unk_idx).sum())} | length {tok.shape[1]}")
_, _, tok = conv([("t", "ACGU")])
assert tok.shape[1] == 6 and (tok == alphabet.unk_idx).sum() == 0, "expected <start> A C G U <end> with no unknown token"

# Edge weights over one window (Brandes et al. 2023): low near the edges, 1 in the middle
i = np.arange(1, W + 1)
w = np.ones(W)
w[i < 256] = 1 / (1 + np.exp(-(i[i < 256] - 128) / 16))
right = i >= W - 256
w[right] = 1 / (1 + np.exp(-((W + 1 - i[right]) - 128) / 16))          # mirrored right edge (assumed symmetric)
WEIGHT = torch.tensor(w, dtype=torch.float32, device=device)

def window_starts(L):
    if L <= W: return [0]
    st = list(range(0, L - W + 1, S))
    if st[-1] != L - W: st.append(L - W)
    return st

@torch.no_grad()
def forward(wins):
    """Per-window nucleotide embeddings (start/end/padding removed), processed BATCH windows at a time."""
    out = []
    for b in range(0, len(wins), BATCH):
        part = wins[b:b + BATCH]
        _, _, tok = conv([(str(k), x) for k, x in enumerate(part)])
        assert (tok == alphabet.unk_idx).sum() == 0, "unknown token: check the T/U convention"
        rep = model(tok.to(device), repr_layers=[LAYER])["representations"][LAYER]
        out += [rep[k, 1:len(x) + 1] for k, x in enumerate(part)]
    return out

def embed_overlap(s):
    """Overlapping windows, edge-weighted per-nucleotide average, then mean pooling."""
    L, st = len(s), window_starts(len(s))
    reps = forward([s[a:a + W] for a in st])
    if len(st) == 1:
        return reps[0].mean(0)
    num = torch.zeros(L, reps[0].shape[1], device=device)
    den = torch.zeros(L, device=device)
    for a, r in zip(st, reps):
        num[a:a + W] += WEIGHT[:, None] * r
        den[a:a + W] += WEIGHT
    return (num / den[:, None]).mean(0)

def embed_nonoverlap(s):
    """Non-overlapping chunks; per-nucleotide embeddings joined, then mean pooling."""
    reps = forward([s[a:a + W] for a in range(0, len(s), W)])
    per_nt = torch.cat(reps)
    assert per_nt.shape[0] == len(s)
    return per_nt.mean(0)

# Self-test and timing on the shortest, a median-length and the longest circRNA
L_ = seq["seq_len"]
pick = [int(L_.idxmin()), int((L_ - L_.median()).abs().idxmin()), int(L_.idxmax())]
for n in pick:
    s = seq["sequence"].iat[n]
    t0 = time.time(); a = embed_overlap(s); dt = time.time() - t0
    b = embed_nonoverlap(s)
    same = torch.allclose(a, b, atol=1e-5)
    print(f"len {len(s):5d} | windows {len(window_starts(len(s)))} | {dt:.2f}s | shape {tuple(a.shape)} | identical: {same}")
    if len(s) <= W:
        assert same, "short circRNAs must give identical embeddings in both versions"
per_win = dt / len(window_starts(int(L_.max())))
print(f"rough total time, overlapping: {per_win * seq['n_windows_overlap'].sum() / 60:.1f} min (overestimate)")

device: cuda
ACGU -> [[0, 4, 5, 6, 7, 2]] | unknown tokens: 0 | length 6
ACGT -> [[0, 4, 5, 6, 3, 2]] | unknown tokens: 1 | length 6
len    75 | windows 1 | 0.02s | shape (640,) | identical: True
len   516 | windows 1 | 0.01s | shape (640,) | identical: True
len  4914 | windows 9 | 0.62s | shape (640,) | identical: False
rough total time, overlapping: 2.9 min (overestimate)


In [19]:
# Cell 1 - overlapping windows for all 1,582 circRNAs
emb = np.zeros((len(seq), 640), dtype=np.float32)
t0 = time.time()
for n, s in enumerate(seq["sequence"]):
    emb[n] = embed_overlap(s).cpu().numpy()
    if n % 100 == 0: print(f"  {n}/{len(seq)}  {time.time() - t0:.0f}s", end="\r")

assert emb.shape == (len(circ), 640) and np.isfinite(emb).all()
np.save(OUT / "rnafm_embeddings_overlap.npy", emb)
seq[["circbase_id", "seq_len", "n_windows_overlap", "n_chunks_nonoverlap"]].to_csv(OUT / "rnafm_embeddings_index.csv", index=False)
(OUT / "rnafm_embeddings_overlap.json").write_text(json.dumps({
    "model": "RNA-FM t12", "layer": LAYER, "window": W, "stride": S,
    "combine": "edge-weighted per-nucleotide average (Brandes et al. 2023)", "pooling": "mean over nucleotides",
    "shape": list(emb.shape), "multi_window": int((seq["n_windows_overlap"] > 1).sum())}, indent=2))
print(f"\nsaved overlap {emb.shape} in {time.time() - t0:.0f}s")

  1500/1582  122s
saved overlap (1582, 640) in 128s


In [20]:
# Cell 2 - non-overlapping chunks; only the 350 long circRNAs need a new pass
emb_non = emb.copy()
long_idx = np.flatnonzero(seq["seq_len"].to_numpy() > W)
for j, n in enumerate(long_idx):
    emb_non[n] = embed_nonoverlap(seq["sequence"].iat[n]).cpu().numpy()
    if j % 50 == 0: print(f"  {j}/{len(long_idx)}", end="\r")

assert len(long_idx) == 350 and np.isfinite(emb_non).all()
np.save(OUT / "rnafm_embeddings_nonoverlap.npy", emb_non)
(OUT/ "rnafm_embeddings_nonoverlap.json").write_text(json.dumps({
    "model": "RNA-FM t12", "layer": LAYER, "window": W, "stride": W,
    "combine": "non-overlapping chunks, nucleotides joined", "pooling": "mean over nucleotides",
    "shape": list(emb_non.shape), "recomputed": int(len(long_idx))}, indent=2))
print(f"\nsaved non-overlap {emb_non.shape}")

  300/350
saved non-overlap (1582, 640)


In [21]:
# Cell 3 - overlap vs non-overlap: same circRNA vs different circRNAs
def cos(a, b):
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))

short = seq["seq_len"].to_numpy() <= W
assert np.allclose(emb[short], emb_non[short]), "short circRNAs should be identical"

same = cos(emb[long_idx], emb_non[long_idx])
other = cos(emb[long_idx], emb[np.random.default_rng(0).permutation(long_idx)])
print(f"same circRNA, two methods:  median cos {np.median(same):.4f} | min {same.min():.4f}")
print(f"different circRNAs (base):  median cos {np.median(other):.4f}")
by_win = pd.Series(same, index=seq["n_windows_overlap"].to_numpy()[long_idx]).groupby(level=0).median().round(4)
print("median cos by number of overlapping windows:\n" + by_win.to_string())

same circRNA, two methods:  median cos 0.9999 | min 0.9991
different circRNAs (base):  median cos 0.9916
median cos by number of overlapping windows:
2    0.9998
3    0.9999
4    0.9999
5    0.9999
6    0.9999
7    1.0000
8    1.0000
9    1.0000


In [22]:
mu = emb.mean(0)
same_c = cos(emb[long_idx] - mu, emb_non[long_idx] - mu)
other_c = cos(emb[long_idx] - mu, emb[np.random.default_rng(0).permutation(long_idx)] - mu)
print(f"centred | same circRNA: median {np.median(same_c):.4f}, min {same_c.min():.4f} | different circRNAs: median {np.median(other_c):.4f}")

centred | same circRNA: median 0.9885, min 0.8332 | different circRNAs: median 0.0608


In [23]:
L_long = seq["seq_len"].to_numpy()[long_idx]
rem = L_long % W
d = pd.DataFrame({
    "circbase_id": seq["circbase_id"].to_numpy()[long_idx],
    "seq_len": L_long,
    "n_windows": seq["n_windows_overlap"].to_numpy()[long_idx],
    "last_chunk_nt": np.where(rem == 0, W, rem),
    "cos_centred": same_c,
})
d["last_chunk_share"] = (d["last_chunk_nt"] / d["seq_len"]).round(3)
print("10 most different:\n" + d.sort_values("cos_centred").head(10).to_string(index=False))
print("\nmedian by number of windows:\n" + d.groupby("n_windows")["cos_centred"].median().round(4).to_string())
print("\nSpearman(last-chunk share, cosine):", round(d[["last_chunk_share", "cos_centred"]].corr("spearman").iloc[0, 1], 3))
print("circRNAs with cosine < 0.95:", int((d["cos_centred"] < 0.95).sum()), "of", len(d))
d.to_csv(OUT / "chunking_comparison.csv", index=False)

10 most different:
     circbase_id  seq_len  n_windows  last_chunk_nt  cos_centred  last_chunk_share
hsa_circ_0004018     1168          2            146     0.833246             0.125
hsa_circ_0082002     1214          2            192     0.838414             0.158
hsa_circ_0088196     1206          2            184     0.838578             0.153
hsa_circ_0030586     1192          2            170     0.850990             0.143
hsa_circ_0035483     1159          2            137     0.860503             0.118
hsa_circ_0068888     1144          2            122     0.864314             0.107
hsa_circ_0038467     1206          2            184     0.891764             0.153
hsa_circ_0001979     1800          3            778     0.897490             0.432
hsa_circ_0070401     1189          2            167     0.899271             0.140
hsa_circ_0043265     1729          3            707     0.905730             0.409

median by number of windows:
n_windows
2    0.9816
3    0.9857
4   

In [24]:
# Cell 4 - layer 6 (pre-registered fallback), overlapping windows only
LAYER = 6                                                    # forward() reads this global at call time
emb6 = np.zeros((len(seq), 640), dtype=np.float32)
t0 = time.time()
for n, s in enumerate(seq["sequence"]):
    emb6[n] = embed_overlap(s).cpu().numpy()
    if n % 100 == 0: print(f"  {n}/{len(seq)}  {time.time() - t0:.0f}s", end="\r")
LAYER = 12                                                   # restore the main layer

assert emb6.shape == (len(circ), 640) and np.isfinite(emb6).all()
np.save(OUT / "rnafm_embeddings_overlap_L6.npy", emb6)
(OUT / "rnafm_embeddings_overlap_L6.json").write_text(json.dumps({
    "model": "RNA-FM t12", "layer": 6, "window": W, "stride": S,
    "combine": "edge-weighted per-nucleotide average (Brandes et al. 2023)", "pooling": "mean over nucleotides",
    "shape": list(emb6.shape), "reason": "pre-registered fallback after miRNA probe not passed"}, indent=2))

# Sanity check: layer 6 must differ from layer 12 (centred cosine, same circRNA)
emb12 = np.load(OUT / "rnafm_embeddings_overlap.npy")
c6, c12 = emb6 - emb6.mean(0), emb12 - emb12.mean(0)
cs = (c6 * c12).sum(1) / (np.linalg.norm(c6, axis=1) * np.linalg.norm(c12, axis=1))
print(f"\nsaved L6 {emb6.shape} in {time.time() - t0:.0f}s | centred cosine L6 vs L12: median {np.median(cs):.3f}")

  1500/1582  121s
saved L6 (1582, 640) in 127s | centred cosine L6 vs L12: median 0.439
